# Document Parsing and Chunk Preparation

This notebook inspects the document-ingestion steps used by the RAG pipeline. It reads the document paths from `config/config.yaml`, parses each file once, filters non-retrievable sections, creates chunks, and optionally writes the intermediate artifacts to `output/v2`.

PDF parsing uploads the configured PDF to MinerU. Run the parsing cell once per notebook session and use the later cells to inspect or export its cached in-memory results. Re-running that cell intentionally submits a new MinerU parsing request.

## Locate the project root

This cell makes imports work when the notebook is opened from any directory within the repository.

In [6]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the project directory tree.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/humengqing/Documents/Code/VSCode/doc-qa-agent


## Load configuration and parser helpers

This cell loads the configured source files, validates that they exist, and defines one parser dispatcher. The dispatcher is the only place in this notebook that selects between the PDF and Word parsers.

In [7]:
import json
from pathlib import Path
from typing import Any

from src.core.config import PROJECT_ROOT, Config
from src.document.chunker import prepare_chunks
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

config = Config()
configured_paths = config.get("documents", "paths", default=[])
if not isinstance(configured_paths, list) or not configured_paths:
    raise ValueError("documents.paths must contain at least one document.")

document_paths: list[Path] = []
for configured_path in configured_paths:
    document_path = Path(configured_path)
    if not document_path.is_absolute():
        document_path = PROJECT_ROOT / document_path
    if not document_path.is_file():
        raise FileNotFoundError(f"Configured document was not found: {document_path}")
    document_paths.append(document_path)


def parse_document(document_path: Path) -> list[dict[str, Any]]:
    suffix = document_path.suffix.lower()
    if suffix == ".pdf":
        return parse_pdf_document(document_path, config)
    if suffix == ".docx":
        return parse_word_document(document_path)
    raise ValueError(f"Unsupported document type: {document_path}")


print("Configured documents:")
for document_path in document_paths:
    print(f"- {document_path.relative_to(PROJECT_ROOT)}")

Configured documents:
- data/raw/HuMengqing.pdf
- data/raw/Report.docx


## Parse each configured document once

This is the only parsing step. It stores parsed sections in `sections_by_document`, so the inspection, chunking, and export cells below reuse the same data without reparsing files.

In [8]:
sections_by_document: dict[str, list[dict[str, Any]]] = {}

for document_path in document_paths:
    sections = parse_document(document_path)
    sections_by_document[document_path.name] = sections
    table_count = sum(section["type"] == "table" for section in sections)
    print(f"{document_path.name}: {len(sections)} sections, {table_count} tables")

HuMengqing.pdf: 84 sections, 10 tables
Report.docx: 104 sections, 23 tables


## Inspect parsed sections

This cell prints a compact inventory of the cached parser output. It includes section type, title, page where available, character count, and the retrieval classification used by the chunking filter.

In [9]:
for source_name, sections in sections_by_document.items():
    print(f"\n{source_name}: {len(sections)} sections")
    for section in sections:
        page_number = section.get("page")
        page_label = str(page_number) if page_number is not None else "-"
        print(
            f"  page {page_label:>3} | {len(section['content']):>6} chars | "
            f"[{section['type']}] {section['title'][:60]} | "
            f"kind={section.get('section_kind', 'body')}"
        )


HuMengqing.pdf: 84 sections
  page   1 |     65 chars | [text] Introduction | kind=front_matter
  page   1 |    406 chars | [text] Development of in-line monitoring of an additive thermoplast | kind=front_matter
  page   1 |    126 chars | [text] Forschungspraktikum Nr.: 85 | kind=front_matter
  page   1 |   2389 chars | [text] Topic: Development of in-line monitoring of an additive ther | kind=front_matter
  page   3 |    396 chars | [text] SELBSTSTÄNDIGKEITSERKLÄRUNG | kind=front_matter
  page   4 |   1278 chars | [text] ABSTRACT | kind=abstract
  page   4 |   1618 chars | [text] CONTENTS | kind=toc
  page   7 |   2588 chars | [text] LIST OF FIGURES | kind=toc
  page   9 |    458 chars | [text] LIST OF TABLES | kind=toc
  page  10 |    994 chars | [table] Table 1 | kind=body
  page  10 |    409 chars | [table] Table 2 | kind=body
  page  11 |   3848 chars | [text] 1 Introduction and Motivation | kind=body
  page  13 |    650 chars | [text] 2 Theoretical Basis and Current Situation |

## Filter sections and prepare chunks

This cell applies the configured exclusion rules and creates retrieval chunks from the cached sections. `prepared_documents` retains both the filtered sections and chunks for the export cell.

In [10]:
prepared_documents: dict[str, dict[str, list[dict[str, Any]]]] = {}
all_chunks: list[dict[str, Any]] = []

for source_name, sections in sections_by_document.items():
    filtered_sections, chunks = prepare_chunks(sections, config)
    prepared_documents[source_name] = {
        "filtered_sections": filtered_sections,
        "chunks": chunks,
    }
    all_chunks.extend(chunks)
    print(
        f"{source_name}: retained {len(filtered_sections)} of {len(sections)} sections; "
        f"created {len(chunks)} chunks"
    )

print(f"Total chunks: {len(all_chunks)}")

HuMengqing.pdf: retained 58 of 84 sections; created 134 chunks
Report.docx: retained 97 of 104 sections; created 172 chunks
Total chunks: 306


## Export cached intermediate artifacts

This optional cell writes the already parsed and chunked data to JSON. It does not call either document parser. The production indexing command remains `python -m scripts.build_index`, which also rebuilds the vector store.

In [11]:
output_root = PROJECT_ROOT / "output" / "v2"
sections_directory = output_root / "sections"
filtered_sections_directory = output_root / "filtered_sections"
chunks_directory = output_root / "chunks"

for output_directory in (
    sections_directory,
    filtered_sections_directory,
    chunks_directory,
):
    output_directory.mkdir(parents=True, exist_ok=True)

for source_name, sections in sections_by_document.items():
    source_stem = Path(source_name).stem
    parsed_path = sections_directory / f"{source_stem}.json"
    filtered_path = filtered_sections_directory / f"{source_stem}.json"
    chunks_path = chunks_directory / f"{source_stem}_chunks.json"

    parsed_path.write_text(json.dumps(sections, ensure_ascii=False, indent=2), encoding="utf-8")
    filtered_path.write_text(
        json.dumps(
            prepared_documents[source_name]["filtered_sections"],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    chunks_path.write_text(
        json.dumps(prepared_documents[source_name]["chunks"], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved cached artifacts for {source_name}")

all_chunks_path = chunks_directory / "all_chunks.json"
all_chunks_path.write_text(
    json.dumps(all_chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved {len(all_chunks)} chunks to {all_chunks_path}")

Saved cached artifacts for HuMengqing.pdf
Saved cached artifacts for Report.docx
Saved 306 chunks to /Users/humengqing/Documents/Code/VSCode/doc-qa-agent/output/v2/chunks/all_chunks.json


## Runtime requirements

The PDF parser requires `MINERU_API_TOKEN` in `.env` or the environment and sends the PDF content to MinerU. Review data-handling requirements before parsing sensitive documents. Word parsing runs locally.